In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
DATA_PATH = Path("../data/interim/Mumbai_AQI_Dataset_Updated_2026.xlsx")

df = pd.read_excel(DATA_PATH)

print(f"Dataset loaded successfully")
print(f"Shape: {df.shape}")

df.head()

Dataset loaded successfully
Shape: (2771, 9)


,City,Date,AQI,PM2.5,PM10,NO2,SO2,CO,O3
0,Mumbai,01/01/2019,194,106.70,209.52,161.02,221.16,2.04,184.30
1,Mumbai,02/01/2019,180,99.00,194.40,149.40,205.20,1.89,171.00
2,Mumbai,03/01/2019,267,146.85,288.36,221.61,304.38,2.80,253.65
3,Mumbai,04/01/2019,223,122.65,240.84,185.09,254.22,2.34,211.85
4,Mumbai,05/01/2019,178,97.90,192.24,147.74,202.92,1.87,169.10


In [3]:
df.columns = [
    "city",
    "date",
    "aqi",
    "pm25",
    "pm10",
    "no2",
    "so2",
    "co",
    "o3"
]

print(df.columns.tolist())

['city', 'date', 'aqi', 'pm25', 'pm10', 'no2', 'so2', 'co', 'o3']


In [4]:
df["date"] = pd.to_datetime(
    df["date"],
    format="%d/%m/%Y",
    errors="coerce"
)

print("Invalid dates:", df["date"].isna().sum())
print("Date range:", df["date"].min(), "to", df["date"].max())

Invalid dates: 0
Date range: 2019-01-01 00:00:00 to 2026-08-06 00:00:00


In [5]:
print("Total missing values:", df.isnull().sum().sum())
print(df.isnull().sum())

Total missing values: 0
city    0
date    0
aqi     0
pm25    0
pm10    0
no2     0
so2     0
co      0
o3      0
dtype: int64


In [6]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [7]:
numeric_columns = [
    "aqi",
    "pm25",
    "pm10",
    "no2",
    "so2",
    "co",
    "o3"
]

negative_values = (df[numeric_columns] < 0).sum()

negative_values

aqi     0
pm25    0
pm10    0
no2     0
so2     0
co      0
o3      0
dtype: int64

In [8]:
invalid_aqi = df[
    (df["aqi"] < 0) |
    (df["aqi"] > 500)
]

print("Invalid AQI records:", len(invalid_aqi))

Invalid AQI records: 0


In [9]:
df = df.sort_values("date").reset_index(drop=True)

print(df[["date", "aqi"]].head(10))

        date  aqi
0 2019-01-01  194
1 2019-01-02  180
2 2019-01-03  267
3 2019-01-04  223
4 2019-01-05  178
5 2019-01-06  192
6 2019-01-07  146
7 2019-01-08  161
8 2019-01-09  154
9 2019-01-10  219


In [10]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["day_of_week"] = df["date"].dt.dayofweek
df["day_of_year"] = df["date"].dt.dayofyear
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

In [11]:
df.head()

,city,date,aqi,pm25,pm10,no2,so2,co,o3,year,month,day,day_of_week,day_of_year,is_weekend
0,Mumbai,2019-01-01,194,106.70,209.52,161.02,221.16,2.04,184.30,2019,1,1,1,1,0
1,Mumbai,2019-01-02,180,99.00,194.40,149.40,205.20,1.89,171.00,2019,1,2,2,2,0
2,Mumbai,2019-01-03,267,146.85,288.36,221.61,304.38,2.80,253.65,2019,1,3,3,3,0
3,Mumbai,2019-01-04,223,122.65,240.84,185.09,254.22,2.34,211.85,2019,1,4,4,4,0
4,Mumbai,2019-01-05,178,97.90,192.24,147.74,202.92,1.87,169.10,2019,1,5,5,5,1


In [12]:
df["aqi_lag_1"] = df["aqi"].shift(1)
df["aqi_lag_3"] = df["aqi"].shift(3)
df["aqi_lag_7"] = df["aqi"].shift(7)

In [13]:
# Define the forecasting target as the next day's AQI
df["target_aqi"] = df["aqi"].shift(-1)

print("Target column created successfully.")
print(df[["date", "aqi", "target_aqi"]].head(10))

Target column created successfully.
        date  aqi  target_aqi
0 2019-01-01  194       180.0
1 2019-01-02  180       267.0
2 2019-01-03  267       223.0
3 2019-01-04  223       178.0
4 2019-01-05  178       192.0
5 2019-01-06  192       146.0
6 2019-01-07  146       161.0
7 2019-01-08  161       154.0
8 2019-01-09  154       219.0
9 2019-01-10  219       212.0


In [14]:
df["aqi_rolling_mean_3"] = (
    df["aqi"]
    .shift(1)
    .rolling(window=3)
    .mean()
)

df["aqi_rolling_mean_7"] = (
    df["aqi"]
    .shift(1)
    .rolling(window=7)
    .mean()
)

df["aqi_rolling_std_7"] = (
    df["aqi"]
    .shift(1)
    .rolling(window=7)
    .std()
)

In [15]:
feature_columns = [
    "date",
    "aqi",
    "aqi_lag_1",
    "aqi_lag_3",
    "aqi_lag_7",
    "aqi_rolling_mean_3",
    "aqi_rolling_mean_7",
    "aqi_rolling_std_7"
]

df[feature_columns].head(10)

,date,aqi,aqi_lag_1,aqi_lag_3,aqi_lag_7,aqi_rolling_mean_3,aqi_rolling_mean_7,aqi_rolling_std_7
0,2019-01-01,194,NaN,NaN,NaN,NaN,NaN,NaN
1,2019-01-02,180,194.0,NaN,NaN,NaN,NaN,NaN
2,2019-01-03,267,180.0,NaN,NaN,NaN,NaN,NaN
3,2019-01-04,223,267.0,194.0,NaN,213.666667,NaN,NaN
4,2019-01-05,178,223.0,180.0,NaN,223.333333,NaN,NaN
5,2019-01-06,192,178.0,267.0,NaN,222.666667,NaN,NaN
6,2019-01-07,146,192.0,223.0,NaN,197.666667,NaN,NaN
7,2019-01-08,161,146.0,178.0,194.0,172.000000,197.142857,38.429280
8,2019-01-09,154,161.0,192.0,180.0,166.333333,192.428571,40.828328
9,2019-01-10,219,154.0,146.0,267.0,153.666667,188.714286,43.257810


In [16]:
lag_columns = [
    "aqi_lag_1",
    "aqi_lag_3",
    "aqi_lag_7",
    "aqi_rolling_mean_3",
    "aqi_rolling_mean_7",
    "aqi_rolling_std_7"
]

df = df.dropna(
    subset=lag_columns + ["target_aqi"]
).reset_index(drop=True)

print("Shape after feature engineering:", df.shape)
print("Remaining missing values:", df.isnull().sum().sum())

Shape after feature engineering: (2763, 22)
Remaining missing values: 0


In [17]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = PROCESSED_DIR / "air_quality_cleaned.csv"

df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Cleaned dataset saved to: {OUTPUT_PATH}")

Cleaned dataset saved to: ..\data\processed\air_quality_cleaned.csv


In [18]:
print("Final shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

df.head()

Final shape: (2763, 22)
Missing values: 0
Duplicate rows: 0


,city,date,aqi,pm25,pm10,no2,so2,co,o3,year,...,day_of_week,day_of_year,is_weekend,aqi_lag_1,aqi_lag_3,aqi_lag_7,target_aqi,aqi_rolling_mean_3,aqi_rolling_mean_7,aqi_rolling_std_7
0,Mumbai,2019-01-08,161,88.55,173.88,133.63,183.54,1.69,152.95,2019,...,1,8,0,146.0,178.0,194.0,154.0,172.000000,197.142857,38.429280
1,Mumbai,2019-01-09,154,84.70,166.32,127.82,175.56,1.62,146.30,2019,...,2,9,0,161.0,192.0,180.0,219.0,166.333333,192.428571,40.828328
2,Mumbai,2019-01-10,219,120.45,236.52,181.77,249.66,2.30,208.05,2019,...,3,10,0,154.0,146.0,267.0,212.0,153.666667,188.714286,43.257810
3,Mumbai,2019-01-11,212,116.60,228.96,175.96,241.68,2.23,201.40,2019,...,4,11,0,219.0,161.0,223.0,200.0,178.000000,181.857143,30.786515
4,Mumbai,2019-01-12,200,110.00,216.00,166.00,228.00,2.10,190.00,2019,...,5,12,1,212.0,154.0,178.0,206.0,195.000000,180.285714,28.534858


In [19]:
print("Final dataframe shape:", df.shape)
print("\nFinal columns:")
print(df.columns.tolist())

print("\nTarget column present:", "target_aqi" in df.columns)

if "target_aqi" in df.columns:
    print("\nTarget AQI preview:")
    print(df["target_aqi"].head(10))

Final dataframe shape: (2763, 22)

Final columns:
['city', 'date', 'aqi', 'pm25', 'pm10', 'no2', 'so2', 'co', 'o3', 'year', 'month', 'day', 'day_of_week', 'day_of_year', 'is_weekend', 'aqi_lag_1', 'aqi_lag_3', 'aqi_lag_7', 'target_aqi', 'aqi_rolling_mean_3', 'aqi_rolling_mean_7', 'aqi_rolling_std_7']

Target column present: True

Target AQI preview:
0    154.0
1    219.0
2    212.0
3    200.0
4    206.0
5    163.0
6    163.0
7    163.0
8    193.0
9    207.0
Name: target_aqi, dtype: float64


In [20]:
print("Final shape:", df.shape)
print("Number of columns:", len(df.columns))
print("Target present:", "target_aqi" in df.columns)
print("Missing values:", df.isnull().sum().sum())

print("\nTarget preview:")
print(df[["date", "aqi", "target_aqi"]].head(10))

print("\nTarget tail:")
print(df[["date", "aqi", "target_aqi"]].tail())

Final shape: (2763, 22)
Number of columns: 22
Target present: True
Missing values: 0

Target preview:
        date  aqi  target_aqi
0 2019-01-08  161       154.0
1 2019-01-09  154       219.0
2 2019-01-10  219       212.0
3 2019-01-11  212       200.0
4 2019-01-12  200       206.0
5 2019-01-13  206       163.0
6 2019-01-14  163       163.0
7 2019-01-15  163       163.0
8 2019-01-16  163       193.0
9 2019-01-17  193       207.0

Target tail:
           date  aqi  target_aqi
2758 2026-08-01   67        46.0
2759 2026-08-02   46        39.0
2760 2026-08-03   39        30.0
2761 2026-08-04   30        57.0
2762 2026-08-05   57        64.0


In [21]:
# Verify the actual saved processed dataset
saved_df = pd.read_csv("../data/processed/air_quality_cleaned.csv")

print("Saved dataset shape:", saved_df.shape)
print("Saved dataset columns:", len(saved_df.columns))
print("target_aqi present:", "target_aqi" in saved_df.columns)
print("Saved dataset missing values:", saved_df.isnull().sum().sum())

Saved dataset shape: (2763, 22)
Saved dataset columns: 22
target_aqi present: True
Saved dataset missing values: 0
